# Modelo de Regresión Lineal basado en productos "mágicos"

In [1]:
# EJECUTAR LIBRERIAS
import os
import pandas as pd

In [2]:
# LEEMOS LOS DATOS
drive_base_path = 'C:/Repositorios-Ing.Carlos-Cicconi/labo3-2025r/Data/'
filename = 'sell-in.txt'
filepath = os.path.join(drive_base_path, filename)
sell = pd.read_csv(filepath, sep='\t')
print(sell.head(10))

filename = 'product_id_apredecir201912.txt'
filepath = os.path.join(drive_base_path, filename)
a_predecir = pd.read_csv(filepath, sep='\t')

   periodo  customer_id  product_id  plan_precios_cuidados  cust_request_qty  \
0   201701        10234       20524                      0                 2   
1   201701        10032       20524                      0                 1   
2   201701        10217       20524                      0                 1   
3   201701        10125       20524                      0                 1   
4   201701        10012       20524                      0                11   
5   201701        10080       20524                      0                 1   
6   201701        10015       20524                      0                 4   
7   201701        10062       20524                      0                 1   
8   201701        10159       20524                      0                 3   
9   201701        10183       20524                      0                 1   

   cust_request_tn       tn  
0          0.05300  0.05300  
1          0.13628  0.13628  
2          0.03028  0.03028  

In [3]:
# Veo la cantidad de valores distintos en "periodo"
num_periodos = sell['periodo'].nunique()
print(f'Cantidad de valores distintos en "periodo": {num_periodos}')

Cantidad de valores distintos en "periodo": 36


In [4]:
# Agrupar y sumar
sell_agrup = (
    sell
    .groupby(['periodo', 'product_id'], as_index=False)['tn']
    .sum()
)

print(sell_agrup)

       periodo  product_id          tn
0       201701       20001   934.77222
1       201701       20002   550.15707
2       201701       20003  1063.45835
3       201701       20004   555.91614
4       201701       20005   494.27011
...        ...         ...         ...
31238   201912       21265     0.05007
31239   201912       21266     0.05121
31240   201912       21267     0.01569
31241   201912       21271     0.00298
31242   201912       21276     0.00892

[31243 rows x 3 columns]


In [5]:
# FEATURE ENGINEERING Y CREACION DE LA CLASE A PREDECIR
# Ordenar por product_id y periodo para asegurar el orden correcto
sell_agrup = sell_agrup.sort_values(['product_id', 'periodo']).reset_index(drop=True)

# Crear las 11 columnas con los valores de los períodos anteriores
for i in range(1, 12):  # Del 1 al 11
    sell_agrup[f'tn_lag_{i}'] = sell_agrup.groupby('product_id')['tn'].shift(i)

# Agregar columna con el valor de tn del período +2 (2 períodos hacia adelante)
sell_agrup['tn_target'] = sell_agrup.groupby('product_id')['tn'].shift(-2)

# Mostrar el resultado
print("Dataset con las 11 columnas de períodos anteriores y 1 columna de período futuro:")
print(sell_agrup.head(36))
print(f"\nForma del dataset: {sell_agrup.shape}")
print(f"Columnas: {list(sell_agrup.columns)}")

Dataset con las 11 columnas de períodos anteriores y 1 columna de período futuro:
    periodo  product_id          tn    tn_lag_1    tn_lag_2    tn_lag_3  \
0    201701       20001   934.77222         NaN         NaN         NaN   
1    201702       20001   798.01620   934.77222         NaN         NaN   
2    201703       20001  1303.35771   798.01620   934.77222         NaN   
3    201704       20001  1069.96130  1303.35771   798.01620   934.77222   
4    201705       20001  1502.20132  1069.96130  1303.35771   798.01620   
5    201706       20001  1520.06539  1502.20132  1069.96130  1303.35771   
6    201707       20001  1030.67391  1520.06539  1502.20132  1069.96130   
7    201708       20001  1267.39462  1030.67391  1520.06539  1502.20132   
8    201709       20001  1316.94604  1267.39462  1030.67391  1520.06539   
9    201710       20001  1439.75563  1316.94604  1267.39462  1030.67391   
10   201711       20001  1580.47401  1439.75563  1316.94604  1267.39462   
11   201712       

In [6]:
# Definir la lista de product_id específicos
magicos = [20002, 20001, 20003, 20004, 20005, 20009, 20006, 20010, 20014, 20007, 20019, 20013, 20011, 20015,
           20008, 20026, 20016, 20023, 20020, 20012, 20017, 20021, 20022, 20018, 20027, 20024, 20025, 20031,
           20042, 20046, 20049, 20028, 20029, 20035, 20045, 20044, 20033, 20054, 20053, 20038, 20039, 20059,
           20084, 20047, 20041, 20061, 20051, 20043, 20075, 20050, 20063, 20116, 20057, 20058, 20073, 20052,
           20065, 20069, 20121, 20070, 20056, 20094, 20081, 20062, 20068, 20071, 20066, 20055, 20076, 20037,
           20112, 20067, 20093, 20030, 20080, 20091, 20107, 20074, 20101, 20095, 20087, 20111, 20120, 20106,
           20145, 20119, 20108, 20077, 20122, 20103, 20158, 20153, 20157, 20123, 20134, 20125, 20118, 20132,
           20155, 20164, 20140, 20099, 20082, 20092, 20072, 20129, 20139, 20148, 20114, 20133, 20096, 20161,
           20227, 20162, 20086, 20144, 20097, 20203, 20100, 20124, 20151, 20146, 20109, 20137, 20167, 20117,
           20090, 20235, 20160, 20079, 20189, 20232, 20166, 20138, 20181, 20142, 20177, 20102, 20212, 20297,
           20175, 20180, 20176, 20184, 20306, 20193, 20196, 20179, 20218, 20316, 20188, 20182, 20201, 20187,
           20224, 20198, 20321, 20205, 20197, 20200, 20152, 20323, 20202, 20220, 20168, 20233, 20208, 20313,
           20215, 20206, 20209, 20238, 20219, 20231, 20222]

# Crear el subconjunto filtrado por período 201812 y los product_id específicos
sell_agrup_subset = sell_agrup[
    (sell_agrup['periodo'] == 201812) & 
    (sell_agrup['product_id'].isin(magicos))
]

print(f"Dataset original shape: {sell_agrup.shape}")
print(f"Subconjunto filtrado shape: {sell_agrup_subset.shape}")
print(f"Cantidad de productos únicos en el subconjunto: {sell_agrup_subset['product_id'].nunique()}")
print("\nPrimeras filas del subconjunto:")
print(sell_agrup_subset.head())

Dataset original shape: (31243, 15)
Subconjunto filtrado shape: (175, 15)
Cantidad de productos únicos en el subconjunto: 175

Primeras filas del subconjunto:
     periodo  product_id          tn    tn_lag_1    tn_lag_2    tn_lag_3  \
23    201812       20001  1486.68669  1813.01511  2295.19832  1438.67455   
59    201812       20002  1009.45458  1766.81068  1378.49032   954.23575   
95    201812       20003   769.82869  1206.91773  1313.34211   912.34156   
131   201812       20004   585.56477   802.34669   809.67086   948.86342   
167   201812       20005   372.63428   469.26344   893.74086   761.77520   

       tn_lag_4    tn_lag_5    tn_lag_6    tn_lag_7    tn_lag_8    tn_lag_9  \
23   1800.96168  1470.41009  1150.79169  1293.89788  1251.28462  1856.83534   
59   1161.88430   977.40239  1033.82845  1103.39191   999.20934   966.86044   
95    955.97079   656.22700   660.73323   784.35885   765.47838   778.55594   
131   936.42001   653.42310   447.84475   641.37063   611.51237   48

In [7]:
# Importar librerías necesarias para regresión lineal
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import numpy as np

# Preparar los datos para el modelo
# Columnas 3 a 14 como predictores (tn_lag_1 hasta tn_lag_11 y tn)
X = sell_agrup_subset.iloc[:, 2:14]  # Columnas 3 a 14 (índices 2 a 13)
# Columna 15 como variable target
y = sell_agrup_subset.iloc[:, 14]    # Columna 15 (índice 14)

print("Variables predictoras (X):")
print(f"Shape: {X.shape}")
print(f"Columnas: {list(X.columns)}")
print("\nVariable target (y):")
print(f"Shape: {y.shape}")
print(f"Nombre: {y.name}")

# Verificar si hay valores NaN y eliminar filas con NaN
print(f"\nValores NaN en X: {X.isnull().sum().sum()}")
print(f"Valores NaN en y: {y.isnull().sum()}")

# Eliminar filas con valores NaN
mask = ~(X.isnull().any(axis=1) | y.isnull())
X_clean = X[mask]
y_clean = y[mask]

print(f"\nDatos después de eliminar NaN:")
print(f"X_clean shape: {X_clean.shape}")
print(f"y_clean shape: {y_clean.shape}")

# Ajustar el modelo de regresión lineal
if len(X_clean) > 0:
    modelo = LinearRegression()
    modelo.fit(X_clean, y_clean)
    
    # Hacer predicciones
    y_pred = modelo.predict(X_clean)
    
    # Calcular métricas
    r2 = r2_score(y_clean, y_pred)
    mse = mean_squared_error(y_clean, y_pred)
    mae = mean_absolute_error(y_clean, y_pred)
    rmse = np.sqrt(mse)
    
    print(f"\n=== RESULTADOS DEL MODELO ===")
    print(f"R² Score: {r2:.4f}")
    print(f"MSE: {mse:.4f}")
    print(f"RMSE: {rmse:.4f}")
    print(f"MAE: {mae:.4f}")
    
    print(f"\n=== COEFICIENTES ===")
    print(f"Intercepto: {modelo.intercept_:.4f}")
    for i, coef in enumerate(modelo.coef_):
        print(f"{X_clean.columns[i]}: {coef:.4f}")
else:
    print("No hay suficientes datos sin NaN para ajustar el modelo.")

Variables predictoras (X):
Shape: (175, 12)
Columnas: ['tn', 'tn_lag_1', 'tn_lag_2', 'tn_lag_3', 'tn_lag_4', 'tn_lag_5', 'tn_lag_6', 'tn_lag_7', 'tn_lag_8', 'tn_lag_9', 'tn_lag_10', 'tn_lag_11']

Variable target (y):
Shape: (175,)
Nombre: tn_target

Valores NaN en X: 0
Valores NaN en y: 0

Datos después de eliminar NaN:
X_clean shape: (175, 12)
y_clean shape: (175,)

=== RESULTADOS DEL MODELO ===
R² Score: 0.9719
MSE: 701.4469
RMSE: 26.4848
MAE: 18.4067

=== COEFICIENTES ===
Intercepto: 11.6888
tn: 0.3188
tn_lag_1: 0.1204
tn_lag_2: 0.1834
tn_lag_3: -0.1056
tn_lag_4: -0.1312
tn_lag_5: -0.0339
tn_lag_6: 0.0646
tn_lag_7: 0.1056
tn_lag_8: 0.0531
tn_lag_9: 0.0211
tn_lag_10: 0.1285
tn_lag_11: 0.1228


In [8]:
# Filtrar sell_agrup por los product_id presentes en a_predecir
sell_agrup_filtrado = sell_agrup[sell_agrup['product_id'].isin(a_predecir['product_id'])]
print(sell_agrup_filtrado)

drive_base_path = 'C:/Repositorios-Ing.Carlos-Cicconi/labo3-2025r/Outputs/Linear Regression'
filename = 'sell_agrup_filtrado.txt'
filepath = os.path.join(drive_base_path, filename)

# Guardar el DataFrame filtrado en un archivo de texto
sell_agrup_filtrado.to_csv(filepath, sep='\t', index=False)

       periodo  product_id          tn    tn_lag_1    tn_lag_2   tn_lag_3  \
0       201701       20001   934.77222         NaN         NaN        NaN   
1       201702       20001   798.01620   934.77222         NaN        NaN   
2       201703       20001  1303.35771   798.01620   934.77222        NaN   
3       201704       20001  1069.96130  1303.35771   798.01620  934.77222   
4       201705       20001  1502.20132  1069.96130  1303.35771  798.01620   
...        ...         ...         ...         ...         ...        ...   
31206   201908       21276     0.01265     0.00223     0.04086    0.09283   
31207   201909       21276     0.01856     0.01265     0.00223    0.04086   
31208   201910       21276     0.02079     0.01856     0.01265    0.00223   
31209   201911       21276     0.03341     0.02079     0.01856    0.01265   
31210   201912       21276     0.00892     0.03341     0.02079    0.01856   

        tn_lag_4  tn_lag_5  tn_lag_6  tn_lag_7  tn_lag_8  tn_lag_9  tn_lag_

In [9]:
# Filtrar los períodos de interés
periodos_interes = [201901, 201902, 201903, 201904, 201905, 201906,
                    201907, 201908, 201909, 201910, 201911, 201912]
sell_agrup_filtrado_pi = sell_agrup_filtrado[sell_agrup_filtrado['periodo'].isin(periodos_interes)]

In [10]:
# Hacer predicciones para todos los product_id en sell_agrup_filtrado_pi
print("=== APLICANDO PREDICCIONES ===")

# Obtener los coeficientes del modelo para usar manualmente
if 'modelo' in locals():
    coef_tn = modelo.coef_[0]  # Coeficiente para 'tn'
    coef_lags = modelo.coef_[1:]  # Coeficientes para tn_lag_1 hasta tn_lag_11
    intercepto = modelo.intercept_
    
    print(f"Coeficiente tn: {coef_tn:.4f}")
    print(f"Coeficientes lag: {coef_lags}")
    print(f"Intercepto: {intercepto:.4f}")
    
    # Crear dataset para predicciones
    predicciones = []
    
    # Obtener todos los product_id únicos
    productos_unicos = sell_agrup_filtrado_pi['product_id'].unique()
    
    for product_id in productos_unicos:
        # Filtrar datos del producto específico
        datos_producto = sell_agrup_filtrado_pi[sell_agrup_filtrado_pi['product_id'] == product_id].copy()
        datos_producto = datos_producto.sort_values('periodo')
        
        # Verificar si tiene datos para todos los 12 períodos (201901 a 201912)
        periodos_completos = [201901, 201902, 201903, 201904, 201905, 201906,
                             201907, 201908, 201909, 201910, 201911, 201912]
        
        periodos_disponibles = set(datos_producto['periodo'].tolist())
        tiene_todos_periodos = all(p in periodos_disponibles for p in periodos_completos)
        
        if tiene_todos_periodos:
            # CASO 1: Usar el modelo de regresión lineal
            # Obtener valores específicos para la predicción manual
            tn_201912 = datos_producto[datos_producto['periodo'] == 201912]['tn'].iloc[0]
            tn_201911 = datos_producto[datos_producto['periodo'] == 201911]['tn'].iloc[0]
            tn_201910 = datos_producto[datos_producto['periodo'] == 201910]['tn'].iloc[0]
            tn_201909 = datos_producto[datos_producto['periodo'] == 201909]['tn'].iloc[0]
            tn_201908 = datos_producto[datos_producto['periodo'] == 201908]['tn'].iloc[0]
            tn_201907 = datos_producto[datos_producto['periodo'] == 201907]['tn'].iloc[0]
            tn_201906 = datos_producto[datos_producto['periodo'] == 201906]['tn'].iloc[0]
            tn_201905 = datos_producto[datos_producto['periodo'] == 201905]['tn'].iloc[0]
            tn_201904 = datos_producto[datos_producto['periodo'] == 201904]['tn'].iloc[0]
            tn_201903 = datos_producto[datos_producto['periodo'] == 201903]['tn'].iloc[0]
            tn_201902 = datos_producto[datos_producto['periodo'] == 201902]['tn'].iloc[0]
            tn_201901 = datos_producto[datos_producto['periodo'] == 201901]['tn'].iloc[0]
            
            # Aplicar la fórmula manual del modelo
            prediccion = (intercepto + 
                         coef_tn * tn_201912 +           # tn actual
                         coef_lags[0] * tn_201911 +      # tn_lag_1
                         coef_lags[1] * tn_201910 +      # tn_lag_2
                         coef_lags[2] * tn_201909 +      # tn_lag_3
                         coef_lags[3] * tn_201908 +      # tn_lag_4
                         coef_lags[4] * tn_201907 +      # tn_lag_5
                         coef_lags[5] * tn_201906 +      # tn_lag_6
                         coef_lags[6] * tn_201905 +      # tn_lag_7
                         coef_lags[7] * tn_201904 +      # tn_lag_8
                         coef_lags[8] * tn_201903 +      # tn_lag_9
                         coef_lags[9] * tn_201902 +      # tn_lag_10
                         coef_lags[10] * tn_201901)      # tn_lag_11
            
            metodo = "Modelo Regresión"
            
        else:
            # CASO 2: Usar promedio de los períodos disponibles
            promedio = datos_producto['tn'].mean()
            prediccion = promedio
            metodo = "Promedio Períodos"
        
        # Agregar resultado
        predicciones.append({
            'product_id': product_id,
            'prediccion_202002': prediccion,
            'metodo_usado': metodo,
            'periodos_disponibles': len(datos_producto),
            'tiene_12_periodos': tiene_todos_periodos
        })
    
    # Crear DataFrame con las predicciones
    df_predicciones = pd.DataFrame(predicciones)
    
    print(f"\n=== RESUMEN DE PREDICCIONES ===")
    print(f"Total productos procesados: {len(df_predicciones)}")
    print(f"Productos con modelo de regresión: {sum(df_predicciones['metodo_usado'] == 'Modelo Regresión')}")
    print(f"Productos con promedio: {sum(df_predicciones['metodo_usado'] == 'Promedio Períodos')}")
    
    print(f"\n=== PRIMERAS PREDICCIONES ===")
    print(df_predicciones.head(10))
    
    # Mostrar estadísticas de las predicciones
    print(f"\n=== ESTADÍSTICAS DE PREDICCIONES ===")
    print(f"Predicción mínima: {df_predicciones['prediccion_202002'].min():.2f}")
    print(f"Predicción máxima: {df_predicciones['prediccion_202002'].max():.2f}")
    print(f"Predicción promedio: {df_predicciones['prediccion_202002'].mean():.2f}")
    print(f"Desviación estándar: {df_predicciones['prediccion_202002'].std():.2f}")
    
else:
    print("Error: El modelo no está disponible. Ejecuta primero la celda anterior.")

=== APLICANDO PREDICCIONES ===
Coeficiente tn: 0.3188
Coeficientes lag: [ 0.12039505  0.18341737 -0.10560963 -0.13118867 -0.03389689  0.06460623
  0.10560896  0.05305297  0.02114199  0.12848177  0.12280457]
Intercepto: 11.6888

=== RESUMEN DE PREDICCIONES ===
Total productos procesados: 780
Productos con modelo de regresión: 650
Productos con promedio: 130

=== PRIMERAS PREDICCIONES ===
   product_id  prediccion_202002      metodo_usado  periodos_disponibles  \
0       20001        1229.049361  Modelo Regresión                    12   
1       20002        1184.794640  Modelo Regresión                    12   
2       20003         763.374105  Modelo Regresión                    12   
3       20004         597.738054  Modelo Regresión                    12   
4       20005         560.294543  Modelo Regresión                    12   
5       20006         473.321652  Modelo Regresión                    12   
6       20007         371.450409  Modelo Regresión                    12   
7 

In [11]:
# Preparar dataset para exportar con las columnas requeridas
df_export = df_predicciones[['product_id', 'prediccion_202002']].copy()

# Renombrar la columna prediccion_202002 a tn
df_export = df_export.rename(columns={'prediccion_202002': 'tn'})

# Mostrar el dataset que se va a exportar
print("Dataset a exportar:")
print(df_export.head(10))
print(f"\nShape del dataset: {df_export.shape}")
print(f"Columnas: {list(df_export.columns)}")

# Definir la ruta de salida
drive_base_path = 'C:/Repositorios-Ing.Carlos-Cicconi/labo3-2025r/Outputs/Linear Regression'
filename = 'predicciones_202002_175.csv'
filepath = os.path.join(drive_base_path, filename)

# Crear el directorio si no existe
os.makedirs(drive_base_path, exist_ok=True)

# Guardar el archivo CSV separado por comas
df_export.to_csv(filepath, sep=',', index=False)

print(f"\nArchivo guardado exitosamente en: {filepath}")
print(f"Total de registros guardados: {len(df_export)}")

# Verificar que el archivo se guardó correctamente
if os.path.exists(filepath):
    file_size = os.path.getsize(filepath)
    print(f"Tamaño del archivo: {file_size} bytes")
    
    # Leer las primeras líneas para verificar el formato
    with open(filepath, 'r') as f:
        primeras_lineas = [f.readline().strip() for _ in range(5)]
    
    print(f"\nPrimeras líneas del archivo CSV:")
    for i, linea in enumerate(primeras_lineas):
        print(f"Línea {i+1}: {linea}")
else:
    print("Error: El archivo no se pudo crear.")

Dataset a exportar:
   product_id           tn
0       20001  1229.049361
1       20002  1184.794640
2       20003   763.374105
3       20004   597.738054
4       20005   560.294543
5       20006   473.321652
6       20007   371.450409
7       20008   368.557927
8       20009   442.357690
9       20010   373.549312

Shape del dataset: (780, 2)
Columnas: ['product_id', 'tn']

Archivo guardado exitosamente en: C:/Repositorios-Ing.Carlos-Cicconi/labo3-2025r/Outputs/Linear Regression\predicciones_202002_175.csv
Total de registros guardados: 780
Tamaño del archivo: 19566 bytes

Primeras líneas del archivo CSV:
Línea 1: product_id,tn
Línea 2: 20001,1229.0493610713581
Línea 3: 20002,1184.7946400327144
Línea 4: 20003,763.3741053329481
Línea 5: 20004,597.7380543091554
